In [1]:
import zipfile
from pathlib import Path

# Put neu-dataset.zip in the same folder as this notebook
zip_path = Path("neu-dataset.zip")
extract_path = Path("dataset")

extract_path.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print("Dataset extracted to:", extract_path.resolve())
print(list(extract_path.iterdir()))


Dataset extracted to: D:\Downloads\dataset
[WindowsPath('dataset/NEU-DET')]


In [2]:
import os

# Adjust this to whichever sub-folder the zip created
root = "dataset"
for dirpath, dirnames, filenames in os.walk(root):
    level = dirpath.replace(root, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(dirpath)}/")
    if level < 3:
        for f in filenames[:3]:
            print(f"{indent}  {f}")

dataset/
  NEU-DET/
    train/
      annotations/
      images/
        crazing/
        inclusion/
        patches/
        pitted_surface/
        rolled-in_scale/
        scratches/
    validation/
      annotations/
      images/
        crazing/
        inclusion/
        patches/
        pitted_surface/
        rolled-in_scale/
        scratches/


In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

BATCH = 32
NUM_WORKERS = 2

transform_224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

transform_299 = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ✅ These folders already have class subfolders inside — ImageFolder works directly
train_data_224 = datasets.ImageFolder("dataset/NEU-DET/train/images",      transform=transform_224)
val_data_224   = datasets.ImageFolder("dataset/NEU-DET/validation/images", transform=transform_224)

train_data_299 = datasets.ImageFolder("dataset/NEU-DET/train/images",      transform=transform_299)
val_data_299   = datasets.ImageFolder("dataset/NEU-DET/validation/images", transform=transform_299)

train_loader_224 = DataLoader(train_data_224, batch_size=BATCH, shuffle=True,  num_workers=NUM_WORKERS)
val_loader_224   = DataLoader(val_data_224,   batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)

train_loader_299 = DataLoader(train_data_299, batch_size=BATCH, shuffle=True,  num_workers=NUM_WORKERS)
val_loader_299   = DataLoader(val_data_299,   batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)

print("Classes:", train_data_224.classes)
print("Train samples:", len(train_data_224), "| Val samples:", len(val_data_224))

Classes: ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
Train samples: 1440 | Val samples: 360


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
from torchvision import models
import torch.nn as nn

NUM_CLASSES = 6

def build_resnet50():
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model.to(device)

def build_inception():
    model = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT, aux_logits=True)
    model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, NUM_CLASSES)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model.to(device)

def build_efficientnet():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
    return model.to(device)

In [4]:
import torch.optim as optim

def train_model(model, train_loader, epochs=10, is_inception=False):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=3e-4)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            if is_inception:
                outputs, aux = model(images)          # InceptionV3 returns (main, aux)
                loss = criterion(outputs, labels) + 0.4 * criterion(aux, labels)
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}]  Loss: {running_loss/len(train_loader):.4f}")


def evaluate_model(model, val_loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            if isinstance(outputs, tuple):
                outputs = outputs[0]
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f"Accuracy: {acc:.2f}%")
    return acc

In [6]:
resnet = build_resnet50()
print("=== ResNet50 ===")
train_model(resnet, train_loader_224, epochs=3)
resnet_acc = evaluate_model(resnet, val_loader_224)

=== ResNet50 ===
Epoch [1/3]  Loss: 0.3369
Epoch [2/3]  Loss: 0.0713
Epoch [3/3]  Loss: 0.0141
Accuracy: 100.00%


In [7]:
inception = build_inception()
print("\InceptionV3 (GoogLeNet)")
train_model(inception, train_loader_299, epochs=3, is_inception=True)
inception_acc = evaluate_model(inception, val_loader_299)

\InceptionV3 (GoogLeNet)
Epoch [1/3]  Loss: 0.3579
Epoch [2/3]  Loss: 0.0614
Epoch [3/3]  Loss: 0.0257
Accuracy: 100.00%


In [8]:
efficientnet = build_efficientnet()
print("\n EfficientNet-B0")
train_model(efficientnet, train_loader_224, epochs=3)
efficientnet_acc = evaluate_model(efficientnet, val_loader_224)



 EfficientNet-B0
Epoch [1/3]  Loss: 0.4202
Epoch [2/3]  Loss: 0.0381
Epoch [3/3]  Loss: 0.0209
Accuracy: 99.72%


In [9]:
print("\n" + "="*40)
print("      MODEL COMPARISON SUMMARY")
print("="*40)
print(f"  ResNet50       : {resnet_acc:.2f}%")
print(f"  InceptionV3    : {inception_acc:.2f}%")
print(f"  EfficientNet-B0: {efficientnet_acc:.2f}%")
print("="*40)


      MODEL COMPARISON SUMMARY
  ResNet50       : 100.00%
  InceptionV3    : 100.00%
  EfficientNet-B0: 99.72%
